# Modelo LSTM: Entrenamiento y Evaluación

In [1]:
import os
import random
import sys
import warnings
from pathlib import Path
import optuna
import time

import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.lstm_wrapper import LSTMModel
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer
from src.data.dataset import TimeSeriesDataset
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "lstm"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any(), "NaNs en train"
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any(), "NaNs en test"
assert df_train.index.max() < df_test.index.min(), "Solapamiento temporal train/test"

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

# Mostramos en qué dispositivo se ejecutará (por si luego lo pasas a Colab)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de ejecución: {device.upper()}")

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)
Dispositivo de ejecución: CPU


## Introducción

La red neuronal de Memoria a Corto y Largo Plazo (*Long Short-Term Memory*, LSTM) es una arquitectura de red neuronal recurrente diseñada para modelar dependencias temporales en datos secuenciales. A diferencia de las redes neuronales tradicionales, incorpora un estado de celda y mecanismos de compuertas entrenables que permiten conservar o descartar información a lo largo del tiempo, resolviendo el problema del desvanecimiento del gradiente.

En el contexto de las series financieras, esta arquitectura ofrece una alternativa a los modelos econométricos tradicionales, al permitir que la relación entre los retornos históricos y la volatilidad futura sea aprendida directamente a partir de los datos. Su estructura recurrente facilita la identificación de patrones temporales y cambios en la dinámica de los retornos, sin establecer previamente una ecuación paramétrica para la evolución de la varianza condicional.

El objetivo de esta sección es presentar, entrenar y evaluar un modelo LSTM para la predicción de la volatilidad del crudo Brent bajo el protocolo del benchmark. Se estudia el efecto de distintas longitudes de secuencia sobre el desempeño predictivo de la red, manteniendo las condiciones de información y evaluación establecidas para los demás modelos del proyecto.

### Arquitectura

Una red LSTM procesa secuencialmente las observaciones de entrada y actualiza un estado de celda (memoria a largo plazo) junto con un estado oculto (memoria a corto plazo). Para controlar este proceso, utiliza tres mecanismos principales: la compuerta de olvido (*forget gate*), la compuerta de entrada (*input gate*) y la compuerta de salida (*output gate*).

Para una observación de entrada $\mathbf{x}_t$ y un estado oculto anterior $\mathbf{h}_{t-1}$, las compuertas se calculan mediante:

$$
\mathbf{f}_t = \sigma\left(\mathbf{W}_f\mathbf{x}_t+\mathbf{U}_f\mathbf{h}_{t-1}+\mathbf{b}_f\right),
$$

$$
\mathbf{i}_t = \sigma\left(\mathbf{W}_i\mathbf{x}_t+\mathbf{U}_i\mathbf{h}_{t-1}+\mathbf{b}_i\right),
$$

$$
\mathbf{o}_t = \sigma\left(\mathbf{W}_o\mathbf{x}_t+\mathbf{U}_o\mathbf{h}_{t-1}+\mathbf{b}_o\right),
$$

donde $\mathbf{f}_t$ representa la compuerta de olvido, $\mathbf{i}_t$ la compuerta de entrada, $\mathbf{o}_t$ la compuerta de salida, $\sigma$ la función sigmoide y $\mathbf{h}_{t-1}$ el estado oculto anterior.

El estado de celda candidato se calcula mediante:

$$
\tilde{\mathbf{C}}_t =
\tanh\left(
\mathbf{W}_C\mathbf{x}_t+
\mathbf{U}_C\mathbf{h}_{t-1}+
\mathbf{b}_C
\right),
$$

El nuevo estado de celda se actualiza combinando la información olvidada y la nueva información:

$$
\mathbf{C}_t = \mathbf{f}_t \odot \mathbf{C}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{C}}_t,
$$

y el nuevo estado oculto se obtiene como:

$$
\mathbf{h}_t = \mathbf{o}_t \odot \tanh\left(\mathbf{C}_t\right),
$$

donde $\odot$ representa el producto de Hadamard (elemento a elemento). 

Si bien la LSTM y la GRU comparten el objetivo de modelar dependencias temporales, presentan diferencias estructurales que condicionan su funcionamiento. La GRU utiliza dos compuertas (actualización y reinicio) y un único estado oculto $(\mathbf{h}_t)$. En contraste, la LSTM emplea tres compuertas (olvido, entrada y salida) y mantiene dos estados separados: el estado de celda $(\mathbf{C}_t)$ y el estado oculto $(\mathbf{h}_t)$. 

Esta diferencia arquitectónica tiene implicaciones funcionales directas. El estado de celda de la LSTM actúa como una memoria a largo plazo que fluye a través del tiempo con mínimas modificaciones lineales, permitiendo retener información de eventos muy distantes en el pasado. La compuerta de salida, a su vez, ofrece un control independiente sobre qué parte de esa memoria se expone en el estado oculto en cada paso. La GRU, al carecer de un estado de celda separado y fusionar las compuertas, posee menos parámetros entrenables. Esto la hace computacionalmente más eficiente y, a menudo, menos propensa al sobreajuste en conjuntos de datos de tamaño moderado, aunque a costa de una menor capacidad teórica para retener dependencias temporales largas.

Una vez calculado el estado oculto final de la secuencia, $\mathbf{h}_t \in \mathbb{R}^{H}$ (donde $H$ es el número de unidades ocultas), este vector resume la información de la ventana temporal. Para generar un pronóstico escalar de la varianza condicional, este vector se proyecta a través de una capa densa (*fully connected layer*) con pesos $\mathbf{w} \in \mathbb{R}^{H}$ y sesgo $b \in \mathbb{R}$:

$$
y_t = \mathbf{w}^{\top}\mathbf{h}_t + b.
$$

Para garantizar que la varianza pronosticada sea estrictamente positiva (un requisito matemático para la volatilidad), el resultado escalar $y_t$ pasa por una función de activación *Softplus*, la cual es una versión suavizada de la función ReLU:

$$
\hat{\sigma}_t^2 = \text{Softplus}(y_t) = \ln(1 + e^{y_t}),
$$

donde $\boldsymbol{\theta}$ agrupa los pesos y sesgos tanto de las unidades LSTM como de la capa de salida.

Estos mecanismos permiten que la red aprenda qué información temporal conservar, modificar o descartar. A diferencia del perceptrón multicapa (*Multilayer Perceptron*, MLP), que recibe los rezagos como un vector de características, la LSTM procesa las observaciones como una secuencia ordenada, incorporando explícitamente la dependencia temporal en su arquitectura.

Para aplicar esta arquitectura al pronóstico de la volatilidad del crudo Brent, sea $r_t$ el retorno logarítmico diario. Para una ventana temporal de longitud $L$, la secuencia de entrada se define como:

$$
\mathbf{X}_t =
\begin{bmatrix}
r_{t-L}\\
r_{t-L+1}\\
\vdots\\
r_{t-1}
\end{bmatrix}
\in \mathbb{R}^{L\times 1}.
$$

El modelo completo (la LSTM y su capa de salida) genera el pronóstico de la varianza condicional del día inmediatamente posterior mediante la función $f_{\boldsymbol{\theta}}(\mathbf{X}_t)$.

El problema se aborda mediante una estrategia de predicción de un paso hacia adelante (*one-step-ahead forecasting*). El entrenamiento consiste en encontrar los parámetros $\boldsymbol{\theta}$ que minimicen la pérdida QLIKE sobre las observaciones disponibles, dado que esta métrica penaliza asimétricamente las subestimaciones de la volatilidad. El problema de optimización se expresa como:

$$
\boldsymbol{\theta}^{*}
=\arg\min_{\boldsymbol{\theta}}
\frac{1}{N}
\sum_{i=1}^{N}
\mathcal{L}_{\text{QLIKE}}
\left(
\sigma_{i}^{2},
\hat{\sigma}_{i}^{2}
\right),
$$

donde $N$ es el número de secuencias de entrenamiento.

Los parámetros se actualizan mediante retropropagación a través del tiempo (*backpropagation through time*, BPTT) y el optimizador Adaptive Moment Estimation (Adam). La elección del número de unidades recurrentes, las capas ocultas, la longitud de las secuencias ($L$) y los parámetros de entrenamiento (tasa de aprendizaje, *dropout*) influye directamente sobre la capacidad de ajuste y generalización del modelo.

La implementación de la LSTM se realiza utilizando la librería `PyTorch` en su versión `2.13.0`, aprovechando su módulo de diferenciación automática para el cálculo de gradientes y la optimización de los pesos.

## Entrenamiento

Para la selección de hiperparámetros del modelo LSTM, se mantuvo el mismo protocolo experimental (Walk-Forward) y el mismo presupuesto computacional descrito en la sección del GRU. Al igual que en dicha arquitectura, la serie temporal se transformó en una matriz de rezagos para predecir el paso inmediatamente posterior, garantizando la integridad temporal y evitando la fuga de información (*look-ahead bias*).

El espacio de búsqueda de Optuna mantuvo los mismos rangos de exploración: la longitud de la ventana de entrada (`look_back`), el número de neuronas en la capa oculta (`hidden_size`), la cantidad de capas recurrentes, la tasa de abandono (`dropout`), la tasa de aprendizaje y la penalización $\ell_2$ (`weight_decay`). El entrenamiento se configuró con un máximo de 200 épocas y una paciencia de 20 para la parada temprana interna. El desempeño se siguió evaluando mediante la pérdida QLIKE sobre los conjuntos de validación.

In [ ]:
def lstm_search_space(trial: optuna.Trial) -> dict:
    return {
        "look_back": trial.suggest_categorical("look_back", [7, 14, 21, 28]),
        "hidden_size": trial.suggest_categorical("hidden_size", [64, 128, 256]), 
        "num_layers": trial.suggest_int("num_layers", 1, 2),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42
    }

exec_id = 1
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=LSTMModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5, warmup=current_warmup,
    embargo=1, n_trials=50,
    suggest_params_fn=lstm_search_space,
    early_stop_rounds=15
)

trainer.run(y_train, rs_train)

El mecanismo de parada temprana detuvo el proceso en 40 evaluaciones, sin alcanzar el límite máximo de 50. Al igual que en el GRU, el mecanismo de poda (*Pruning*) no descartó ninguna configuración (0 podados), lo que indicó que todas las arquitecturas evaluadas completaron las particiones de validación sin inestabilidad numérica. La mejor configuración registró un QLIKE de 0.7257.

El entrenamiento bajo el esquema Walk-Forward tuvo una duración de 9.6 minutos (573.54 segundos), con un tiempo medio de 2.87 segundos por pliegue. En comparación con el GRU, el cual requirió 430.66 segundos (1.72 segundos por pliegue), el LSTM necesitó aproximadamente un 33 % más de tiempo de cómputo. Este incremento reflejó el costo adicional de la arquitectura LSTM, la cual emplea tres compuertas y un estado de celda separado, en contraste con las dos compuertas del GRU. Frente a los modelos paramétricos (GARCH: 9.12 s; EGARCH: 12.54 s), el costo computacional de la LSTM fue sustancialmente mayor, evidenciando el compromiso entre el tiempo de ejecución y la flexibilidad no paramétrica.

In [4]:
resumen_data = []
exec_dir = RESULTS_PATH 

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_qlike = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
    
db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(study_name="LSTMModel_study", storage=storage_url)

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

        mejor_qlike = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
40,40,0,0.7257,573.5400,2.8680


El ranking de configuraciones muestra que las cinco mejores opciones presentaron una pérdida QLIKE en el rango de 0.726 a 0.764. Las diferencias entre estas configuraciones fueron menores que la alta variabilidad entre folds ($\sigma \approx 0.48$ a $0.51$), lo que indicó un empate estadístico en su desempeño predictivo. A pesar de que los hiperparámetros continuos (tasa de aprendizaje, *dropout* y regularización L2) variaron, se observó un consenso estricto en la arquitectura base: todas seleccionaron una ventana de memoria (*lookback*) de 7 días, 256 neuronas ocultas y una sola capa recurrente. 

Este resultado contrasta con la arquitectura seleccionada para el GRU (21 días y 64 neuronas). La convergencia de la LSTM hacia una ventana de 7 días (equivalente a una semana de negociación) sugiere que esta red priorizó la información de muy corto plazo. Por otro lado, la elección de 256 neuronas es un hallazgo relevante, ya que corresponde al límite superior del espacio de búsqueda definido. Esto indica que el modelo requirió la máxima capacidad paramétrica disponible para intentar capturar la dinámica del Brent. A diferencia de la parsimonia observada en el GRU, la LSTM compensó su mayor complejidad estructural (tres compuertas y estado de celda) incrementando el tamaño de la capa oculta. Sin embargo, esta combinación de alta capacidad y ventana corta resultó en una mayor varianza entre folds, lo que evidenció una mayor propensión al sobreajuste en comparación con el GRU.

In [5]:
df_lstm = pd.read_csv(RESULTS_PATH / "LSTMModel_results.csv")

df_lstm["lr"] = df_lstm["learning_rate"].apply(lambda x: f"{x:.4f}")
df_lstm["wd"] = df_lstm["weight_decay"].apply(lambda x: f"{x:.1e}")

cols_agrupar = ["look_back", "hidden_size", "num_layers", "dropout", "lr", "wd"]
df_agg = df_lstm.groupby(cols_agrupar)["qlike"].agg(["mean", "std"]).reset_index()
df_agg["QLIKE"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)
df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "look_back": "Lookback",
    "hidden_size": "Neuronas",
    "num_layers": "Capas",
    "dropout": "Dropout",
    "lr": "LR",
    "wd": "Weight Decay"
})

df_final = df_top5[["Lookback", "Neuronas", "Capas", "Dropout", "LR", "Weight Decay", "QLIKE"]]

tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,Neuronas,Capas,Dropout,LR,Weight Decay,QLIKE
7,256,1,0.012,0.0004,4.1e-06,0.726 ± 0.487
7,256,1,0.477,0.0002,5.2e-06,0.736 ± 0.476
7,256,1,0.306,0.0002,2.4e-06,0.745 ± 0.514
7,256,1,0.494,0.0002,3.3e-06,0.752 ± 0.513
7,256,1,0.498,0.0002,3.5e-06,0.764 ± 0.507


Las dos configuraciones con menor pérdida compartieron la misma arquitectura base (Lookback de 7 días, 256 neuronas y 1 capa), difiriendo únicamente en sus hiperparámetros continuos. La diferencia entre sus medias ($\Delta \text{QLIKE} = 0.010$) fue menor que la variabilidad observada entre los folds ($\sigma > 0.47$), lo que indicó que ambas fueron indistinguibles en desempeño predictivo.

Un hallazgo relevante fue la magnitud de la desviación estándar del QLIKE en la LSTM ($\sigma \approx 0.48$), la cual superó a la reportada por el GRU ($\sigma \approx 0.30$) y por los modelos GARCH y EGARCH ($\sigma \approx 0.06$ y $0.10$, respectivamente). Una hipótesis plausible para explicar este fenómeno radica en la naturaleza de los algoritmos de optimización. Mientras que el GARCH utiliza una estimación determinista de Máxima Verosimilitud que converge a un único óptimo, la LSTM se entrena mediante descenso de gradiente estocástico sobre una superficie de pérdida no convexa. Esto, sumado a la inicialización aleatoria de los pesos y a la mayor cantidad de parámetros de la LSTM frente al GRU, indujo una mayor varianza en los resultados ante los distintos regímenes de volatilidad presentes en cada fold.

Ante la equivalencia predictiva en la media, se aplicó el criterio de estabilidad y desempeño. Se seleccionó la primera configuración, la cual alcanzó el menor QLIKE medio y utilizó un *dropout* de 0.012 con una tasa de aprendizaje de 4.0e-04 y una regularización L2 de 4.1e-06, representando la variante más estable dentro de la arquitectura dominante.

In [24]:
best_model = LSTMModel(
    look_back=7, hidden_size=256,
    num_layers=1, dropout=0.012, learning_rate=0.0004,
    weight_decay=4.1e-06, max_epochs=200, patience=20, retrain_freq=1, seed=42
)

best_model.fit(y_train, rs_train)

El análisis de la curva de aprendizaje descartó la presencia de sobreajuste clásico. En un escenario de sobreajuste típico, la pérdida de entrenamiento continuaría disminuyendo mientras la de validación empezaría a incrementarse, abriendo una brecha progresiva. En este caso, la pérdida de validación no solo se mantuvo estable, sino que fue consistentemente menor que la de entrenamiento desde el inicio. Esta brecha negativa se explicó por el sesgo de selección temporal del conjunto de datos: la partición cronológica destinó el periodo de mayor dificultad predictiva (los shocks extremos de 2020) al entrenamiento, mientras que la validación (2021-2022) correspondió a un régimen de volatilidad más moderado y, por tanto, con un QLIKE intrínsecamente más bajo.

In [25]:
tt.plot_learning_curve(
    history=best_model.get_history(), 
    title="Curva de Aprendizaje",
    ylab = "Quasi-Likelihood (QLIKE)",
    xlab_pad=14, ylab_pad=14,
    line_width=2.0,
    title_size=14, label_size=14, 
    tick_size=12, legend_size=14,
    ylim=(0, 8), xlim=(0, 26), y_step=0.5, x_step=1,
    legend_bbox=(0.95, 0.95)
)

El modelo alcanzó su punto de mínimo error en la época 5. Esta convergencia temprana indicó que la red encontró rápidamente un mínimo local que no pudo superar en las 20 épocas posteriores, a pesar de la paciencia configurada. Este comportamiento reveló una característica del proceso de optimización de la arquitectura seleccionada (una sola capa con 256 neuronas). Aunque la red utilizó el límite superior de capacidad del espacio de búsqueda, el error de validación se estancó tras la época 5. Esto sugirió que añadir mayor capacidad paramétrica no garantizó una mejora continua del proceso de aprendizaje para este conjunto de datos, sino que la red encontró un nivel de error basal que no pudo superar.

In [20]:
best_model.net.eval()
look_back = best_model.context_length

train_ds = TimeSeriesDataset(y_train, rs_train * best_model.target_scale, look_back)
train_loader = DataLoader(train_ds, batch_size=len(train_ds), shuffle=False)

with torch.no_grad():
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(best_model.device)
        preds_train_var = best_model.net(x_batch).cpu().numpy().flatten()

preds_train_var = preds_train_var / best_model.target_scale
y_train_aligned = y_train[look_back:]
rs_train_aligned = rs_train[look_back:]

In [26]:
print(best_model.net)

LSTMNetwork(
  (lstm): LSTM(1, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)


In [21]:
std_residuals_train = y_train_aligned / np.sqrt(preds_train_var)
std_residuals_train = std_residuals_train[np.isfinite(std_residuals_train)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_train,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),18.2804,.050,.151,Sin Autocorrelación
Ljung-Box (Lag 20),52.0767,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),75.5818,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),216.1327,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),308.8688,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),473.2733,< .001,< .001,Efectos ARCH Significativos
Jarque-Bera,22424.7090,< .001,--,Residuos No Normales
Lilliefors,0.0929,< .001,--,Residuos No Normales
BDS (Dim 2),11.7477,< .001,--,Estructura No Lineal Significativa


In [7]:
preds_train_var

array([0.00045358, 0.00045362, 0.00045361, ..., 0.00045371, 0.00045365,
       0.00045365], shape=(2164,), dtype=float32)

In [8]:
rs_train[look_back:]

array([4.82877585e-05, 9.23407933e-05, 1.01166188e-05, ...,
       3.92774877e-04, 2.25162730e-04, 1.24320475e-04], shape=(2164,))

In [22]:
look_back = best_model.context_length

dates_aligned = df_train.index[look_back:]
rs_volatilidad_aligned = np.sqrt(rs_train[look_back:])
pred_volatilidad = np.sqrt(preds_train_var) 

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,      
    pred_variance=pred_volatilidad,
    title="Ajuste dentro la Muestra de Entrenamiento (LSTM)", 
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad LSTM $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35), 
    legend_bbox=(0.01, 0.98), 
    zoom_slice=mask_crisis, 
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14, label_size=14, tick_size=12, legend_size=14
)